In [53]:
import os
import copy
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print(f"Using Device : {device}")

if device.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")
    print(f"GPU Count    : {torch.cuda.device_count()}")
else:
    print("Running on CPU")

print("=" * 50)

Using Device : cuda
GPU          : NVIDIA GeForce RTX 3050 Laptop GPU
CUDA Version : 12.8
GPU Count    : 1


In [55]:
# ==========================================================
# Paths
# ==========================================================

TRAIN_DIR = "../data/raw/Lung_Disease_Dataset/train"
VAL_DIR = "../data/raw/Lung_Disease_Dataset/val"

MODEL_PATH = "../models/Transfer_1/efficientnet_head_best.pth"

FINETUNE_DIR = "../models/Transfer_2"
os.makedirs(FINETUNE_DIR, exist_ok=True)

# ==========================================================
# Dataset
# ==========================================================

IMAGE_SIZE = 224
NUM_CLASSES = 5

# ==========================================================
# Training Configuration
# ==========================================================

# RTX 3050 (4GB VRAM)
BATCH_SIZE = 16

# Fine-tuning learning rates
BACKBONE_LR = 5e-6
HEAD_LR = 3e-5

WEIGHT_DECAY = 1e-4

NUM_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 7

# Gradient clipping
MAX_GRAD_NORM = 1.0

# Scheduler
LR_FACTOR = 0.5
LR_PATIENCE = 2
MIN_LR = 1e-7

# Random seed
SEED = 42

In [56]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # Safe augmentations for chest X-rays
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(degrees=5),

    transforms.RandomAffine(
        degrees=5,
        translate=(0.02, 0.02),
        scale=(0.98, 1.02),
        shear=2
    ),

    transforms.ColorJitter(
        brightness=0.03,
        contrast=0.03
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [57]:
# ==========================================================
# Load Datasets
# ==========================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform
)

# ==========================================================
# Dataset Information
# ==========================================================

print("=" * 60)
print("Dataset Summary")
print("=" * 60)

print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(val_dataset)}")
print(f"Number of Classes : {len(train_dataset.classes)}")

print("\nClasses:")
for idx, class_name in enumerate(train_dataset.classes):
    print(f"{idx}: {class_name}")

print("\nClass to Index Mapping:")
print(train_dataset.class_to_idx)

print("=" * 60)

Dataset Summary
Training Images   : 6054
Validation Images : 2016
Number of Classes : 5

Classes:
0: Bacterial Pneumonia
1: Corona Virus Disease
2: Normal
3: Tuberculosis
4: Viral Pneumonia

Class to Index Mapping:
{'Bacterial Pneumonia': 0, 'Corona Virus Disease': 1, 'Normal': 2, 'Tuberculosis': 3, 'Viral Pneumonia': 4}


In [58]:
from collections import Counter

train_counts = Counter(train_dataset.targets)
val_counts = Counter(val_dataset.targets)

print("\nTraining Set Distribution")
for idx, class_name in enumerate(train_dataset.classes):
    print(f"{class_name:<22}: {train_counts[idx]}")

print("\nValidation Set Distribution")
for idx, class_name in enumerate(train_dataset.classes):
    print(f"{class_name:<22}: {val_counts[idx]}")


Training Set Distribution
Bacterial Pneumonia   : 1205
Corona Virus Disease  : 1218
Normal                : 1207
Tuberculosis          : 1220
Viral Pneumonia       : 1204

Validation Set Distribution
Bacterial Pneumonia   : 401
Corona Virus Disease  : 406
Normal                : 402
Tuberculosis          : 406
Viral Pneumonia       : 401


In [59]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [60]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    drop_last=False
)

In [61]:
model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(

    nn.Dropout(0.5),

    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),

    nn.Dropout(0.30),

    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),

    nn.Dropout(0.20),

    nn.Linear(256, NUM_CLASSES)

)

model = model.to(device)

In [62]:
# ==========================================================
# Load Stage 3 Head-Trained Model
# ==========================================================

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Checkpoint not found:\n{MODEL_PATH}")

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

print("=" * 60)
print("✅ Stage 3 Head-Trained Model Loaded Successfully")
print("=" * 60)

print(f"Checkpoint Path : {MODEL_PATH}")
print(f"Best Epoch      : {checkpoint.get('epoch', 'N/A')}")
print(f"Validation Loss : {checkpoint.get('val_loss', 'N/A'):.4f}" if isinstance(checkpoint.get('val_loss'), (float, int)) else f"Validation Loss : {checkpoint.get('val_loss', 'N/A')}")
print(f"Accuracy        : {checkpoint.get('accuracy', 'N/A')}")
print(f"Precision       : {checkpoint.get('precision', 'N/A')}")
print(f"Recall          : {checkpoint.get('recall', 'N/A')}")
print(f"F1 Score        : {checkpoint.get('f1', 'N/A')}")

print("=" * 60)

✅ Stage 3 Head-Trained Model Loaded Successfully
Checkpoint Path : ../models/Transfer_1/efficientnet_head_best.pth
Best Epoch      : 35
Validation Loss : N/A
Accuracy        : 0.8333333333333334
Precision       : 0.8320405068161879
Recall          : 0.832641498721153
F1 Score        : 0.8286508544743258


In [63]:
# ==========================================================
# Freeze / Unfreeze Layers
# ==========================================================

# Freeze entire EfficientNet backbone
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze last EfficientNet blocks
UNFREEZE_BLOCKS = [6, 7, 8]

for block in UNFREEZE_BLOCKS:
    for param in model.features[block].parameters():
        param.requires_grad = True

# Keep classifier fully trainable
for param in model.classifier.parameters():
    param.requires_grad = True

In [64]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable Params: {trainable:,}")
print(f"Total Params: {total:,}")
print(f"Percentage: {100*trainable/total:.2f}%")

for name, child in model.features.named_children():
    requires_grad = any(p.requires_grad for p in child.parameters())
    print(f"features[{name}] trainable: {requires_grad}")

Trainable Params: 3,945,761
Total Params: 4,797,569
Percentage: 82.25%
features[0] trainable: False
features[1] trainable: False
features[2] trainable: False
features[3] trainable: False
features[4] trainable: False
features[5] trainable: False
features[6] trainable: True
features[7] trainable: True
features[8] trainable: True


In [65]:
print("=" * 60)
print("Trainable Parameters")
print("=" * 60)

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()
        print(f"✓ {name}")

print("=" * 60)
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Percentage : {100 * trainable_params / total_params:.2f}%")
print("=" * 60)

Trainable Parameters
✓ features.6.0.block.0.0.weight
✓ features.6.0.block.0.1.weight
✓ features.6.0.block.0.1.bias
✓ features.6.0.block.1.0.weight
✓ features.6.0.block.1.1.weight
✓ features.6.0.block.1.1.bias
✓ features.6.0.block.2.fc1.weight
✓ features.6.0.block.2.fc1.bias
✓ features.6.0.block.2.fc2.weight
✓ features.6.0.block.2.fc2.bias
✓ features.6.0.block.3.0.weight
✓ features.6.0.block.3.1.weight
✓ features.6.0.block.3.1.bias
✓ features.6.1.block.0.0.weight
✓ features.6.1.block.0.1.weight
✓ features.6.1.block.0.1.bias
✓ features.6.1.block.1.0.weight
✓ features.6.1.block.1.1.weight
✓ features.6.1.block.1.1.bias
✓ features.6.1.block.2.fc1.weight
✓ features.6.1.block.2.fc1.bias
✓ features.6.1.block.2.fc2.weight
✓ features.6.1.block.2.fc2.bias
✓ features.6.1.block.3.0.weight
✓ features.6.1.block.3.1.weight
✓ features.6.1.block.3.1.bias
✓ features.6.2.block.0.0.weight
✓ features.6.2.block.0.1.weight
✓ features.6.2.block.0.1.bias
✓ features.6.2.block.1.0.weight
✓ features.6.2.block.1.1.

In [66]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(
    [
        {"params": model.features[6].parameters(), "lr": BACKBONE_LR},
        {"params": model.features[7].parameters(), "lr": BACKBONE_LR},
        {"params": model.features[8].parameters(), "lr": BACKBONE_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.3,
    patience=2,
    min_lr=1e-7
)

In [67]:
from copy import deepcopy

# ==========================================================
# Training History
# ==========================================================

history = {
    "train_loss": [],
    "val_loss": [],

    "train_acc": [],
    "val_acc": [],

    "train_precision": [],
    "val_precision": [],

    "train_recall": [],
    "val_recall": [],

    "train_f1": [],
    "val_f1": [],

    "learning_rate": [],
    "medical_score": []
}

# ==========================================================
# Medical Score (Balanced)
# ==========================================================

RECALL_WEIGHT = 0.60
F1_WEIGHT = 0.40

# ==========================================================
# Best Model Tracking
# ==========================================================

best_score = 0.0
best_epoch = 0

best_val_loss = float("inf")
best_val_acc = 0.0
best_val_precision = 0.0
best_val_recall = 0.0
best_val_f1 = 0.0

# ==========================================================
# Early Stopping
# ==========================================================

early_stop_counter = 0

# ==========================================================
# Best Model Weights
# ==========================================================

best_model_weights = deepcopy(model.state_dict())

best_model_path = os.path.join(
    FINETUNE_DIR,
    "efficientnet_finetuned_best.pth"
)

In [68]:
# ============================================================
# Sanity Check Before Fine-Tuning
# ============================================================

model.eval()

sanity_preds = []
sanity_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        sanity_preds.extend(predicted.cpu().numpy())
        sanity_labels.extend(labels.numpy())

sanity_acc = accuracy_score(sanity_labels, sanity_preds)
sanity_precision = precision_score(
    sanity_labels,
    sanity_preds,
    average="macro"
)
sanity_recall = recall_score(
    sanity_labels,
    sanity_preds,
    average="macro"
)
sanity_f1 = f1_score(
    sanity_labels,
    sanity_preds,
    average="macro"
)

medical_score = (
    RECALL_WEIGHT * sanity_recall +
    F1_WEIGHT * sanity_f1
)

print("="*60)
print("Stage 3 Checkpoint Verification")
print("="*60)
print(f"Accuracy        : {sanity_acc:.4f}")
print(f"Macro Precision : {sanity_precision:.4f}")
print(f"Macro Recall    : {sanity_recall:.4f}")
print(f"Macro F1        : {sanity_f1:.4f}")
print(f"Medical Score   : {medical_score:.4f}")

best_score = medical_score
best_epoch = 0
best_model_weights = deepcopy(model.state_dict())

Stage 3 Checkpoint Verification
Accuracy        : 0.8328
Macro Precision : 0.8314
Macro Recall    : 0.8321
Macro F1        : 0.8281
Medical Score   : 0.8305


In [69]:
# ==========================================================
# Training Timer
# ==========================================================
training_start_time = time.time()

# ==========================================================
# Fine-Tuning
# ==========================================================
for epoch in range(NUM_EPOCHS):
    epoch_start_time = time.time()

    print("\n" + "=" * 60)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
    print("=" * 60)

    # ======================================================
    # TRAIN
    # ======================================================
    model.train()
    running_loss = 0.0
    train_labels = []
    train_preds = []

    train_bar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}", leave=False)

    for images, labels in train_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

        optimizer.step()

        running_loss += loss.item()
        predictions = outputs.argmax(dim=1)

        train_labels.extend(labels.cpu().numpy())
        train_preds.extend(predictions.cpu().numpy())

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds) * 100
    train_precision = precision_score(train_labels, train_preds, average="macro", zero_division=0)
    train_recall = recall_score(train_labels, train_preds, average="macro", zero_division=0)
    train_f1 = f1_score(train_labels, train_preds, average="macro", zero_division=0)

    # ======================================================
    # VALIDATION
    # ======================================================
    model.eval()
    running_loss = 0.0
    val_labels = []
    val_preds = []

    val_bar = tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            predictions = outputs.argmax(dim=1)

            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(predictions.cpu().numpy())

            val_bar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss = running_loss / len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds) * 100
    macro_precision = precision_score(val_labels, val_preds, average="macro", zero_division=0)
    macro_recall = recall_score(val_labels, val_preds, average="macro", zero_division=0)
    macro_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    # ======================================================
    # MEDICAL SCORE
    # ======================================================
    medical_score = (RECALL_WEIGHT * macro_recall) + (F1_WEIGHT * macro_f1)

    scheduler.step(medical_score)

    current_lr = optimizer.param_groups[-1]["lr"]

    # ======================================================
    # HISTORY
    # ======================================================
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    history["train_precision"].append(train_precision)
    history["val_precision"].append(macro_precision)

    history["train_recall"].append(train_recall)
    history["val_recall"].append(macro_recall)

    history["train_f1"].append(train_f1)
    history["val_f1"].append(macro_f1)

    history["learning_rate"].append(current_lr)
    history["medical_score"].append(medical_score)

    # ======================================================
    # PRINT METRICS
    # ======================================================
    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.2f}%")
    print(f"Train Precision : {train_precision:.4f}")
    print(f"Train Recall    : {train_recall:.4f}")
    print(f"Train F1        : {train_f1:.4f}")
    print()
    print(f"Val Loss        : {val_loss:.4f}")
    print(f"Val Accuracy    : {val_acc:.2f}%")
    print(f"Val Precision   : {macro_precision:.4f}")
    print(f"Val Recall      : {macro_recall:.4f}")
    print(f"Val F1          : {macro_f1:.4f}")
    print()
    print(f"Medical Score   : {medical_score:.4f}")
    print(f"Learning Rate   : {current_lr:.2e}")

    # ======================================================
    # SAVE BEST MODEL
    # ======================================================
    if medical_score > best_score:
        best_score = medical_score
        best_epoch = epoch + 1
        early_stop_counter = 0
        best_model_weights = deepcopy(model.state_dict())

        best_val_loss = val_loss
        best_val_acc = val_acc
        best_val_precision = macro_precision
        best_val_recall = macro_recall
        best_val_f1 = macro_f1

        torch.save(
            {
                "epoch": epoch + 1,

                # Model
                "model_state_dict": model.state_dict(),

                # Optimizer
                "optimizer_state_dict": optimizer.state_dict(),

                # Scheduler
                "scheduler_state_dict": scheduler.state_dict(),

                # Validation Metrics
                "val_loss": best_val_loss,
                "accuracy": best_val_acc,
                "precision": best_val_precision,
                "recall": best_val_recall,
                "f1": best_val_f1,
                "medical_score": best_score,

                # Training Metrics
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "train_precision": train_precision,
                "train_recall": train_recall,
                "train_f1": train_f1,

                # Learning Rate
                "learning_rate": current_lr,
            },
            best_model_path,
        )

        print("\n✅ Best model updated!")
        print(f"Medical Score : {best_score:.4f}")

    else:
        early_stop_counter += 1
        print(f"\nNo Improvement ({early_stop_counter}/{EARLY_STOPPING_PATIENCE})")

    # ======================================================
    # EPOCH TIME
    # ======================================================
    epoch_time = time.time() - epoch_start_time
    print(f"Epoch Time     : {epoch_time:.1f} sec")

    # ======================================================
    # EARLY STOPPING
    # ======================================================
    if early_stop_counter >= EARLY_STOPPING_PATIENCE:
        print("\n🛑 Early stopping triggered!")
        break


Epoch [1/25]


Training Epoch 1:   0%|          | 0/378 [00:00<?, ?it/s]

Train Loss      : 0.8010
Train Accuracy  : 78.12%
Train Precision : 0.7776
Train Recall    : 0.7807
Train F1        : 0.7786

Val Loss        : 0.7504
Val Accuracy    : 80.26%
Val Precision   : 0.8036
Val Recall      : 0.8018
Val F1          : 0.7953

Medical Score   : 0.7992
Learning Rate   : 3.00e-05

No Improvement (1/7)
Epoch Time     : 108.5 sec

Epoch [2/25]


Train Loss      : 0.7755
Train Accuracy  : 79.86%
Train Precision : 0.7946
Train Recall    : 0.7981
Train F1        : 0.7958

Val Loss        : 0.7250
Val Accuracy    : 81.94%
Val Precision   : 0.8178
Val Recall      : 0.8187
Val F1          : 0.8134

Medical Score   : 0.8166
Learning Rate   : 3.00e-05

No Improvement (2/7)
Epoch Time     : 89.4 sec

Epoch [3/25]


Train Loss      : 0.7641
Train Accuracy  : 80.74%
Train Precision : 0.8042
Train Recall    : 0.8067
Train F1        : 0.8050

Val Loss        : 0.7273
Val Accuracy    : 81.55%
Val Precision   : 0.8163
Val Recall      : 0.8147
Val F1          : 0.8093

Medical Score   : 0.8126
Learning Rate   : 3.00e-05

No Improvement (3/7)
Epoch Time     : 95.9 sec

Epoch [4/25]


Train Loss      : 0.7460
Train Accuracy  : 81.68%
Train Precision : 0.8138
Train Recall    : 0.8163
Train F1        : 0.8146

Val Loss        : 0.7251
Val Accuracy    : 81.70%
Val Precision   : 0.8119
Val Recall      : 0.8162
Val F1          : 0.8104

Medical Score   : 0.8139
Learning Rate   : 3.00e-05

No Improvement (4/7)
Epoch Time     : 98.0 sec

Epoch [5/25]


Train Loss      : 0.7555
Train Accuracy  : 80.97%
Train Precision : 0.8070
Train Recall    : 0.8092
Train F1        : 0.8078

Val Loss        : 0.7340
Val Accuracy    : 81.40%
Val Precision   : 0.8170
Val Recall      : 0.8132
Val F1          : 0.8073

Medical Score   : 0.8108
Learning Rate   : 9.00e-06

No Improvement (5/7)
Epoch Time     : 98.3 sec

Epoch [6/25]


Train Loss      : 0.7361
Train Accuracy  : 82.08%
Train Precision : 0.8176
Train Recall    : 0.8202
Train F1        : 0.8186

Val Loss        : 0.7075
Val Accuracy    : 82.64%
Val Precision   : 0.8254
Val Recall      : 0.8257
Val F1          : 0.8211

Medical Score   : 0.8238
Learning Rate   : 9.00e-06

No Improvement (6/7)
Epoch Time     : 97.8 sec

Epoch [7/25]


Train Loss      : 0.7401
Train Accuracy  : 81.89%
Train Precision : 0.8161
Train Recall    : 0.8184
Train F1        : 0.8170

Val Loss        : 0.7182
Val Accuracy    : 81.40%
Val Precision   : 0.8157
Val Recall      : 0.8132
Val F1          : 0.8070

Medical Score   : 0.8107
Learning Rate   : 9.00e-06

No Improvement (7/7)
Epoch Time     : 96.8 sec

🛑 Early stopping triggered!


NO Improvement Can be seen ....